# SciPlex LFC Heldout Molecules

PCA is excluded because it is an idealized baseline. Each colored point is one cell line; black diamonds show the mean across cell lines.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

repo = Path.cwd()
while not (repo / "results" / "scores" / "sciplex_lfc_heldout_molecules_fresh.csv").exists() and repo != repo.parent:
    repo = repo.parent
score_path = repo / "results" / "scores" / "sciplex_lfc_heldout_molecules_fresh.csv"
fig_dir = repo / "results" / "figures" / "fig_A8_sciplex_lfc_heldout_molecules"
fig_dir.mkdir(parents=True, exist_ok=True)
with open(repo / "results" / "metadata" / "fig_index.json") as f:
    fig_index = json.load(f)

mpl.rcParams.update(fig_index.get("mpl_params", {}))
cell_palette = {"A549": "#4C78A8", "K562": "#F58518", "MCF7": "#54A24B"}
cell_order = ["A549", "K562", "MCF7"]
label_map = {
    "morgan_initialized_lpm": "Morgan-initialized LPM",
    "random": "Random Embeddings",
    "ChemBERTa-77M-MLM": "ChemBERTa-77M-MLM",
    "ChemBERTa-77M-MTR": "ChemBERTa-77M-MTR",
    "chatgpt": "ChatGPT",
    "maccs": "MACCS",
    "topological": "Topological",
    "secfp": "SECFP",
    "avalon": "Avalon",
    "erg": "ErG",
    "ecfp": "ECFP",
    "fcfp": "FCFP",
    "cats2D": "CATS2D",
    "cats3D": "CATS3D",
}

df = pd.read_csv(score_path)
df = df[~df["embedding"].isin(["pca", "ECFP:2_pkl"])].copy()
df["Display name"] = df["embedding"].map(lambda x: label_map.get(x, x))
order = df.groupby("Display name")["L2"].mean().sort_values().index.tolist()

for estimator, title in [("knn", "KNN"), ("lasso", "LASSO")]:
    d = df[df["estimator"] == estimator].copy()
    estimator_order = [name for name in order if name in set(d["Display name"])]
    fig, ax = plt.subplots(figsize=(8.5, 7.0), constrained_layout=True)
    sns.stripplot(
        data=d, y="Display name", order=estimator_order, x="L2",
        hue="cell_line", hue_order=cell_order, palette=cell_palette,
        dodge=False, jitter=0.18, alpha=0.9, s=7, ax=ax,
    )
    means = d.groupby("Display name", observed=True)["L2"].mean().reindex(estimator_order)
    y_positions = {name: i for i, name in enumerate(estimator_order)}
    ax.scatter(means.values, [y_positions[name] for name in estimator_order], marker="D", s=42, color="black", label="Mean", zorder=10)
    best_name = means.idxmin()
    ax.axvline(means.loc[best_name], color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.grid(axis="y")
    ax.set_title(f"SciPlex LFC Heldout Molecules ({title})")
    ax.set_xlabel("L2")
    ax.set_ylabel(None)
    handles, labels = ax.get_legend_handles_labels()
    seen = set(); unique = []
    for h, lab in zip(handles, labels):
        if lab not in seen:
            seen.add(lab); unique.append((h, lab))
    ax.legend([h for h, _ in unique], [lab for _, lab in unique], title=None, frameon=False, loc="lower right")
    for ext in ["png", "pdf"]:
        fig.savefig(fig_dir / f"sciplex_lfc_heldout_{estimator}.{ext}", dpi=220, bbox_inches="tight")
    plt.show()
